# Interactive Dashboard Analysis

We will follow this tutorial by @atishjn to look at building interactive plots and basic dashboard is to create the widgets. The tutorial walks through the  creation of interactive widgets and functions using interact or layout.

Many of the plots used in this notebook leverages [hvplot](https://hvplot.holoviz.org/en/docs/latest/index.html), to create interactive pandas-like plots

In [ ]:
# !pip install hvplot

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import panel as pn
import geopandas
import hvplot.pandas
from datetime import datetime
import folium
from panel.interact import interact
pn.extension()
pn.extension('tabulator')

In [ ]:
# Need to run pn.extension() to render dashboards
pn.extension("plotly",
             "tabulator", # for displaying dataframes
            #  sizing_mode="stretch_width" # for folium
             )

In [ ]:
## Load in dataset
df = pd.read_csv('https://raw.githubusercontent.com/CS133-DataVisualization/CS133_SP26_datafile/refs/heads/main/Module06_interactive-plots/master.csv')

In [ ]:
df.head(2)

,country,year,sex,age,suicides_no,population,suicides/100k pop,country-year,HDI for year,gdp_for_year ($),gdp_per_capita ($),generation
0,Albania,1987,male,15-24 years,21,312900,6.71,Albania1987,NaN,"2,156,624,900",796,Generation X
1,Albania,1987,male,35-54 years,16,308000,5.19,Albania1987,NaN,"2,156,624,900",796,Silent


In [ ]:
## Explore the data
# df.nunique()
# df.country.value_counts()
# df.isna().sum()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27820 entries, 0 to 27819
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             27820 non-null  object 
 1   year                27820 non-null  int64  
 2   sex                 27820 non-null  object 
 3   age                 27820 non-null  object 
 4   suicides_no         27820 non-null  int64  
 5   population          27820 non-null  int64  
 6   suicides/100k pop   27820 non-null  float64
 7   country-year        27820 non-null  object 
 8   HDI for year        8364 non-null   float64
 9   gdp_for_year ($)    27820 non-null  int64  
 10  gdp_per_capita ($)  27820 non-null  int64  
 11  generation          27820 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.5+ MB


In [ ]:
## Some data modification
# Strip whitespace from all column names .str.strip()
df.columns = df.columns.str.strip()

# Replace the ',' with empty string to convert str to int
df['gdp_for_year ($)']=df['gdp_for_year ($)'].str.replace(',','').astype(str).astype(int)

In [ ]:
# Make DataFrame interactive
idf = df.interactive()
idf.head(3)

In [ ]:
# Define Panel widgets for Year
year_slider = pn.widgets.IntSlider(
                                  name=<name of slider>
                                  , start=<int>
                                  , end=<int>
                                  , step=<int>
                                  , value=<default start>
                                  , tooltips=True
                                  )
year_slider

IntSlider(end=2016, name='Year slider', start=1985, step=2, value=2000)

In [ ]:
year_level_agg=df.pivot_table(index='year',
                              values=['suicides_no', 'population', 'gdp_for_year ($)'],
                              aggfunc={'suicides_no':sum,'population':sum,'gdp_for_year ($)':sum}).reset_index()
year_level_agg.head()

,year,gdp_for_year ($),population,suicides_no
0,1985,110964754234500,1008600086,116063
1,1986,132609631982868,1029909613,120670
2,1987,155769882274200,1095029726,126842
3,1988,175557691204476,1054094424,121026
4,1989,191618261647176,1225514347,160244


In [ ]:
## Calculation for new columns
year_level_agg['suicides per 100k people']=np.round(year_level_agg['suicides_no']*100000/year_level_agg['population'],2)
year_level_agg['gdp per capita($)']=np.round((year_level_agg['gdp_for_year ($)']/year_level_agg['population']).astype(float), 2)

## Check datatypes
# year_level_agg.dtypes

In [ ]:
year_level_agg.head()

,year,gdp_for_year ($),population,suicides_no,suicides per 100k people,gdp per capita($)
0,1985,110964754234500,1008600086,116063,11.51,110018.58
1,1986,132609631982868,1029909613,120670,11.72,128758.51
2,1987,155769882274200,1095029726,126842,11.58,142251.74
3,1988,175557691204476,1054094424,121026,11.48,166548.35
4,1989,191618261647176,1225514347,160244,13.08,156357.42


In [ ]:
# Make DataFrame interactive
iyear_level_agg = year_level_agg.interactive()

Set up the `RadioButtonGroup` widget to make selection 'suicides per 100k people' and 'gdp per capita($)' (calculated above).

In [ ]:
# Radio buttons for suicide rate measures
yaxis_year_world = pn.widgets.RadioButtonGroup(
    name='Y axis',
    options=<selections for buttons>,
    button_type='success'
)

# Interative pipeline output applying radio button selection
world_year_pipeline = (
    iyear_level_agg[
        (iyear_level_agg.year <= year_slider) # set cap of dataframe output
    ]
    .groupby(['year'])[yaxis_year_world].mean() # values for table
    .to_frame()
    .reset_index()
    .sort_values(by=['year'])
    .reset_index(drop=True)
)
world_year_pipeline

In [ ]:
### hvplot line plot to look at selection filtering on year
world_year_pipeline_plot = <interactive pipeline>.hvplot(x='year'
                                                      , y=<RadioButtonGroup selection>
                                                      , line_width=3
                                                      , title="Worldwide Suicides & GDP per Capita by Year"
                                                      , width=800, height=400
                                                      )
world_year_pipeline_plot

In [ ]:
### Year-Gender view
year_gender_level_agg = df.pivot_table(index=['year','sex']
                                       , values=['suicides_no', 'population', 'gdp_for_year ($)']
                                       , aggfunc={'suicides_no':sum, 'population':sum, 'gdp_for_year ($)':sum}
                                       ).reset_index()

## Calculation for new columns
year_gender_level_agg['suicides per 100k people']=np.round(year_gender_level_agg['suicides_no']*100000/year_gender_level_agg['population'], 2)
year_gender_level_agg['gdp per capita($)']=np.round((year_gender_level_agg['gdp_for_year ($)']/year_gender_level_agg['population']).astype(float), 0)

# year_gender_level_agg.head()
# year_gender_level_agg.dtypes

In [ ]:
# Make DataFrame interactive
iyear_gender_level_agg = year_gender_level_agg.interactive()

In [ ]:
world_year_gender_pipeline = (
    iyear_gender_level_agg[
        (iyear_gender_level_agg.year <= year_slider)
    ]
    .groupby(['year','sex'])[yaxis_year_world].mean()
    .to_frame()
    .reset_index()
    .sort_values(by=['year','sex'])
    .reset_index(drop=True)
)

In [ ]:
world_year_gender_pipeline = <interactive pipeline>.hvplot(x=<col from interactive table>
                                                               , by=<col to groupby>
                                                               , y=<Radiobutton selection>
                                                               , line_width=2
                                                               , title="Worldwide Suicides & GDP per Capita by Gender-Year"
                                                               , width=800, height=400
                                                               , legend='top'
                                                               )
world_year_gender_pipeline

In [ ]:
### Year-Age focus
year_age_level_agg=df.pivot_table(index=['year', 'age']
                                  , values=['suicides_no', 'population', 'gdp_for_year ($)']
                                  , aggfunc={'suicides_no':sum,'population':sum,'gdp_for_year ($)':sum}
                                  ).reset_index()

## Calculation for new columns
year_age_level_agg['suicides per 100k people']=np.round(year_age_level_agg['suicides_no']*100000/year_age_level_agg['population'],2)
year_age_level_agg['gdp per capita($)']=np.round((year_age_level_agg['gdp_for_year ($)']/year_age_level_agg['population']).astype(float),0)

# print(year_age_level_agg.head())
# print(year_age_level_agg.dtypes)

## Convert dataframe to interactive
iyear_age_level_agg = year_age_level_agg.interactive()

year_age_level_agg.head()

,year,age,gdp_for_year ($),population,suicides_no,suicides per 100k people,gdp per capita($)
0,1985,15-24 years,18494125705750,196974439,17870,9.07,93891.0
1,1985,25-34 years,18494125705750,173536624,20771,11.97,106572.0
2,1985,35-54 years,18494125705750,246046628,35748,14.53,75165.0
3,1985,5-14 years,18494125705750,199192522,984,0.49,92845.0
4,1985,55-74 years,18494125705750,152769432,28736,18.81,121059.0


In [ ]:
## Interactive pipeline to control output based on year
world_year_age_pipeline = (
    iyear_age_level_agg[
        (iyear_age_level_agg.year <= year_slider)
    ]
    .groupby(['year','age'])[yaxis_year_world].mean()
    .to_frame()
    .reset_index()
    .sort_values(by=['year','age'])
    .reset_index(drop=True)
)

world_year_age_pipeline.head()

In [ ]:
world_year_age_pipeline = world_year_age_pipeline.hvplot(x='year'
                                                         , by='age'
                                                         , y=yaxis_year_world
                                                         , line_width=2
                                                         , title="Worldwide Suicides & GDP per Capita by Age-Year"
                                                         , width=800, height=400
                                                         , legend='top'
                                                         )

In [ ]:
### Interactive pipeline wtih age suicide rate/100k
world_age_pipeline = (
    idf
    .groupby(['age'])['suicides/100k pop'].mean().rename("% Suicides per 100k People").transform(lambda x: 100*x/x.sum())
    .to_frame()
    .reset_index()
    .sort_values(by=['% Suicides per 100k People'])
    .reset_index(drop=True)
)


In [ ]:
### Bar plot
world_age_pipeline_plot = world_age_pipeline.hvplot.bar(x=<col to groupby>
                                                        , y=<values for y-axis>
                                                        , rot=45 # rotate xlabel
                                                        , width=500, height=250
                                                        , title="Age wise %Suicides per 100k: 1985-2016"
                                                        , legend=False # remove legend
                                                        , cmap='Pastel2'
                                                        , color=<col to color via grouby>
                                                        )
world_age_pipeline_plot

## Question 16.1: Create an interactive pipeline that separates on a variable
Create an interactive pipeline that uses the values calculated below to get build the suicides_per_sex dataframe. Have the pipeline use the suicides_per_sex DataFrame and `sort_value(by=['sex'])`

In [ ]:
### Your code . . .
suicides_per_sex = idf.groupby(['sex'])['suicides/100k pop'].mean().rename("% Suicides per 100k People").transform(lambda x: 100*x/x.sum())
suicides_per_sex

## Question 16.2: Draw a bar plot
Use the `gender_suicide_pipeline` to draw a `hvplot.bar()` that display the count of "% Suicides per 100k People" separated by gender (`'sex'`).

In [ ]:
### Your code . . .

In [ ]:
## List of unique country names
country_list = list(df['country'].unique())

# Create dropdown widget to Select country
select = pn.widgets.Select(name='Select Country', options=country_list)

In [ ]:
### Interactive pipeline to incorporate country selection and year filtering
country_year_pipeline = (
    idf[(idf['country']==select)&(idf['year']<= year_slider)]
    .groupby(['year'])['suicides/100k pop'].mean().rename("Suicides per 100k People").transform(lambda x: round(x,2))
    .to_frame()
    .reset_index()
    .sort_values(by=['year'])
    .reset_index(drop=True)
)

country_year_pipeline

In [ ]:
country_year_pipeline.hvplot.table(width=500)

### Panel to display multiple components
Combine multiple interaction pipeline and their parameters to combine multiple components.

Below we will look at two types of elements that we can interact with using the same interaction pipelines, 'country_year_pipeline'.

In [ ]:
country_stats = pn.panel(<1st interaction pipeline>.hvplot.table(
                                                        width=500
                                                        , title="Data Table"
                                                        )+<2nd interaction pipeline>.hvplot.line(x='year', y='Suicides per 100k People', cmap='Pastel2'
                                                        , rot=0, width=800, height=400, title="Avg Suicides per 100k people", legend=False)
                                                        )

country_stats

Interactive(Interactive)

## Question 16.3: Build a panel with multiple components
Create a panel that combines two interactive pipelines; `gender_suicide_pipeline` and `country_year_pipeline` to plot a bar plot and line plot

In [ ]:
### Your code . . .

Interactive(Interactive)

In [ ]:
# Create widget with Tabs()
tabs = pn.Tabs()

tabs.extend([('Worldwide Suicides & GDP per capita', world_year_pipeline_plot),
             ('By Gender Worldwide Suicides & GDP per capita', world_year_gender_pipeline),
            ('By Age Worldwide Suicides & GDP per capita', world_year_age_pipeline)
            ])

tabs

Tabs
    [0] Interactive(Interactive, name='Worldwide Suicides &...)
    [1] Interactive(Interactive, name='By Gender Worldwide S...)
    [2] Interactive(Interactive, name='By Age Worldwide S...)